## Root Cause Analysis - Exp1

* Goals
  * Root Cause Analysis on individual RAG results --> 2 RAGs results comparison
  * Provide actionable suggestions
* About
  * Update Auto Eval logic
  * Compare auto eval results before & after the changes

### Auto Eval Output - before changes

In [9]:
%load_ext autoreload
%autoreload 2

import os
import psycopg2
import pandas as pd

from utils_root_cause import *

import warnings
warnings.filterwarnings('ignore')

# check saved output
DATABASE_URL_PUBLIC = os.getenv("DATABASE_URL_PUBLIC_RAG_DR")
conn = psycopg2.connect(DATABASE_URL_PUBLIC)
conn.autocommit = True
db_name = conn.get_dsn_parameters()['dbname']
print(f"Connected to database: {db_name}")
cur = conn.cursor()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Connected to database: railway


In [2]:
cur.execute("""
    SELECT count(distinct config_hash) from existing_rag_output
""")
rows = cur.fetchall()
print(rows[0][0])
print()

cur.execute("""
    SELECT count(distinct config_hash) from existing_auto_eval_output
""")
rows = cur.fetchall()
print(rows[0][0])

39

32


In [3]:
cur.execute("""
    SELECT
        t1.config_hash,
        t1.dataset,
        t1.embedding_model,
        t1.top_n_retrieval,
        t1.semantic_weight,
        t1.answer_gen_llm,
        rq.value->>'query' AS query,
        rq.value->>'context' AS context,
        rq.value->>'retrieved_content' AS retrieved_content,
        rq.value->>'same_context' AS same_context,
        rq.value->>'retrieval_quality_score' AS retrieval_quality_score,
        rq.value->>'rq_reasoning' AS rq_reasoning,
        aq.value->>'referenced_answer' AS referenced_answer,
        aq.value->>'ai_answer' AS ai_answer,
        aq.value->>'answer_quality_score' AS answer_quality_score,
        aq.value->>'aq_reasoning' AS aq_reasoning
    FROM existing_rag_output AS t1
    JOIN existing_auto_eval_output AS t2
        ON t1.config_hash = t2.config_hash,
    jsonb_array_elements(t2.retrieval_quality) WITH ORDINALITY AS rq(value, idx),
    jsonb_array_elements(t2.answer_quality) WITH ORDINALITY AS aq(value, idx)
    WHERE rq.idx = aq.idx
""")
rows = cur.fetchall()
col_names = [desc[0] for desc in cur.description]
df = pd.DataFrame(rows, columns=col_names)
print(df.shape)
df.head()

(960, 16)


,config_hash,dataset,embedding_model,top_n_retrieval,semantic_weight,answer_gen_llm,query,context,retrieved_content,same_context,retrieval_quality_score,rq_reasoning,referenced_answer,ai_answer,answer_quality_score,aq_reasoning
0,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,true,3,The retrieved content is essentially identical...,Have the check reissued to the proper payee.Ju...,To deposit a cheque issued to an associate in ...,2,The AI’s answer is relevant to the user’s ques...
1,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,true,3,The retrieved content fully matches the contex...,Sure you can. You can fill in whatever you wa...,"Yes, you can fill in your business name and ad...",2,The AI’s answer confirms that a business can u...
2,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,Sure you can. You can fill in whatever you wa...,false,2,The retrieved content acknowledges that a DBA ...,You're confusing a lot of things here. Company...,You can have one Employer Identification Numbe...,-1,The AI’s answer directly addresses the user’s ...
3,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...,Set up a meeting with the bank that handles yo...,true,3,The retrieved content is identical to the cont...,"""I'm afraid the great myth of limited liabilit...",Establishing a relationship with a banker is c...,2,The AI answer addresses the user’s request and...
4,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...,The time horizon for your 401K/IRA is essentia...,true,3,The retrieved content is essentially identical...,You should probably consult an attorney. Howev...,If your employer's 401(k) plan is terminated d...,2,The AI’s answer acknowledges that a 401(k) pla...


In [7]:
google_drive_folder_id = '1DLyuFLm0N6BwR2NvDoCEPPMHTzUK5xuv'
output_filename = 'rag_eval_output_v1.xlsx'
upload_to_google_drive(df, google_drive_folder_id, output_filename)

Overwritten file ID: 1XMeCrpFXT3kVIsR4d8VIfSFMZc5mJ-yi


In [4]:
cur.close()
conn.close()

In [5]:
pd.DataFrame(df[['retrieval_quality_score', 'answer_quality_score']].value_counts())

count
retrieval_quality_score answer_quality_score       
3                       2                       405
                        3                       126
0                       2                        97
3                       1                        61
                        4                        60
2                       -1                       36
3                       -1                       24
2                       2                        23
1                       2                        21
2                       1                        17
0                       1                        16
3                       0                        15
2                       4                        12
0                       4                        11
1                       -1                        8
                        1                         8
                        4                         4
0                       -1                        4
                        0                         4
                        3                         3
1                       3                         2
2                       0                         2
                        3                         1

### Auto Eval Output - after changes

In [6]:
from datasets import load_dataset


fiqa_eval = load_dataset("explodinggradients/fiqa", "ragas_eval")['baseline']

rag_lst = []
for idx, record in enumerate(fiqa_eval):
    context = ''.join(record['contexts'])
    gt = ''.join(record['ground_truths'])
    if 'answer' in record.keys():
        ai0_answer = record['answer'].strip()
    else:
        ai0_answer = None

    rag_lst.append({
        'question': record['question'],
        'context': context,
        'context_ct': len(record['contexts']),
        'ground_truth': gt,
        'ai0_answer': ai0_answer
    })

rag_df = pd.DataFrame(rag_lst)
print(rag_df.shape, max(rag_df['context_ct']), min(rag_df['context_ct']))
rag_df.head()

(30, 5) 1 1


,question,context,context_ct,ground_truth,ai0_answer
0,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,1,Have the check reissued to the proper payee.Ju...,The best way to deposit a cheque issued to an ...
1,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,1,Sure you can. You can fill in whatever you wa...,"Yes, you can send a money order from USPS as a..."
2,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,1,You're confusing a lot of things here. Company...,"Yes, it is possible to have one EIN doing busi..."
3,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...,1,"""I'm afraid the great myth of limited liabilit...",Applying for and receiving business credit can...
4,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...,1,You should probably consult an attorney. Howev...,If your employer has closed and you need to tr...


In [7]:
import os
from sqlalchemy import make_url
import warnings
warnings.filterwarnings('ignore')


DATABASE_URL_PUBLIC = os.getenv("DATABASE_URL_PUBLIC_RAG_DR")
DATABASE_URL = DATABASE_URL_PUBLIC.replace("postgres://", "postgresql://")
db_url = make_url(DATABASE_URL)

In [10]:
config_hashes = df['config_hash'].unique().tolist()
eval_results = await run_auto_eval(config_hashes, db_url, rag_df)
print(len(eval_results))

32


In [17]:
new_df_lst = []
for config_hash, eval_result in eval_results.items():
    new_df_sub = eval_result['retrieval_quality_df']\
        .merge(eval_result['answer_quality_df'][['query', 'answer_quality_score', 'aq_reasoning']],
                                         on=['query'])
    new_df_sub['config_hash'] = config_hash
    new_df_sub['dataset'] = 'FIQA Data'
    new_df_lst.append(new_df_sub)

new_df = pd.concat(new_df_lst, ignore_index=True)
print(new_df.shape)
new_df.head()

(960, 12)


,query,ai_answer,referenced_answer,retrieved_content,context,retrieval_quality_score,rq_reasoning,same_context,answer_quality_score,aq_reasoning,config_hash,dataset
0,How to deposit a cheque issued to an associate...,To deposit a cheque issued to an associate in ...,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,3,The retrieved content is essentially identical...,True,2,The AI’s answer is relevant to the user’s ques...,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data
1,Can I send a money order from USPS as a business?,"Yes, you can fill in your business name and ad...",Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,3,The retrieved content exactly matches the cont...,True,3,The AI answer correctly states that a business...,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data
2,1 EIN doing business under multiple business n...,You can have one Employer Identification Numbe...,You're confusing a lot of things here. Company...,Sure you can. You can fill in whatever you wa...,You're confusing a lot of things here. Company...,2,The retrieved content is relevant to the query...,False,-1,The referenced answer discusses merging LLCs a...,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data
3,Applying for and receiving business credit,Establishing a relationship with a banker is c...,"""I'm afraid the great myth of limited liabilit...",Set up a meeting with the bank that handles yo...,Set up a meeting with the bank that handles yo...,3,The retrieved content is highly relevant to th...,True,2,The AI’s answer is relevant to the user’s ques...,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data
4,401k Transfer After Business Closure,If your employer's 401(k) plan is terminated d...,You should probably consult an attorney. Howev...,The time horizon for your 401K/IRA is essentia...,The time horizon for your 401K/IRA is essentia...,3,The retrieved content is essentially identical...,True,2,The AI’s answer acknowledges that a 401(k) can...,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data


In [19]:
pd.DataFrame(new_df[['retrieval_quality_score', 'answer_quality_score']].value_counts())

count
retrieval_quality_score answer_quality_score       
3                        2                      294
                         3                      176
0                        2                       76
3                        0                       52
                         4                       49
4                        2                       42
3                        1                       37
2                       -1                       33
0                        1                       29
3                       -1                       26
2                        2                       21
4                        1                       16
0                        4                       15
1                        2                       14
2                        1                       14
                         4                       10
4                        3                        8
                         0                        8
1                        1                        7
                        -1                        7
0                        0                        5
                         3                        4
1                        4                        4
4                       -1                        3
2                        3                        3
1                        0                        2
2                        0                        2
4                        4                        2
1                        3                        1

In [20]:
google_drive_folder_id = '1DLyuFLm0N6BwR2NvDoCEPPMHTzUK5xuv'
output_filename = 'rag_eval_output_v2.xlsx'
upload_to_google_drive(new_df, google_drive_folder_id, output_filename)

Uploaded file ID: 1uYGrDFUMfTSk3Bmw-p9EpMNHq2HGmNA4
